[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/orientation/mechanical_work/mechanical_work.ipynb)

# Mechanical work in a mirror

Mechanical work measures how an applied force drives a motion. A mirrored crank turns the other way, but the mirrored push supplies the same power.

Projective geometric algebra represents the handle and force direction as points, and the motion rate as a bivector. Joining the handle to the force gives the load that acts on the crank.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np
from numga import NumpyContext, stack
from numga.algebras import PGA2D
from examples.animation import show_animation
from examples.orientation.mechanical_work import render

In [ ]:
ga = PGA2D
mv = NumpyContext(ga).multivector
# + extrinsic, - intrinsic
Point = ga.gatype.antivector()                  # Antivector-
Plane = ga.gatype.vector()                     # Vector+
Rate = ga.gatype.bivector()                    # Bivector+
Load = ga.gatype.antibivector()                # Antibivector-
Scalar = ga.gatype.scalar()                    # Scalar+
Motor = ga.gatype.motor()                      # Even+

A tangential push follows each handle as the crank turns. Pairing its load with the motion rate measures the power delivered. The same computation applies to both sides of the mirror.

In [ ]:
separation = 1.65
radius = 0.9
speed = 1
force_strength = 1.5
steps = 36
duration_ms = 85
times = np.linspace(0, 2 * np.pi / speed, steps, endpoint=False)
mirror = mv.x                                               # Vector+
pivot = mv.point([-separation, 0])                           # Antivector-
handle = mv.point([-separation + radius, 0])                 # Antivector-
force = mv.direction([0, force_strength])                   # Antivector-

# These two planes meet at the pivot and generate its rotation.
rate = ((mv.x + separation * mv.w) ^ mv.y) * speed           # Bivector+

# Reflect the physical ingredients before computing either crank's power.
pivots = stack([pivot, mirror >> pivot])                    # [cranks] Antivector-
handles = stack([handle, mirror >> handle])                 # [cranks] Antivector-
forces = stack([force, mirror >> force])                    # [cranks] Antivector-
rates = stack([rate, mirror >> rate])                       # [cranks] Bivector+

poses = (rates[None, :] * (-times[:, None] / 2)).exp()         # [steps, cranks] Even+
handles = poses >> handles                                 # [steps, cranks] Antivector-
forces = poses >> forces                                   # [steps, cranks] Antivector-

# The force acts at the moving handle, whose velocity follows from the rate.
velocities = handles.commutator(rates)                      # [steps, cranks] Antivector-
loads = handles & forces                                   # [steps, cranks] Antibivector-
powers = loads & rates                                     # [steps, cranks] Scalar+

print("Crank power", powers[0, 0].coefficient_table(), sep="\n")
print("Mirrored crank power", powers[0, 1].coefficient_table(), sep="\n")
show_animation(render.animate(pivots, handles, velocities, forces), "mechanical_work", duration_ms)

Teal arrows show handle velocity, and orange arrows show applied force. The dashed line is the mirror. Both pushes act along the motion and supply the same positive power throughout the turn. The load is an intrinsically oriented vector and the rate is an extrinsically oriented bivector; their regressive pairing is an extrinsic scalar, unchanged by reflection. Their different grades do not replace that orientation information.